# 01 · Modules, Packages, Imports & `if __name__ == "__main__"`

**Module 08 – Modules, Packages & Environments** · Estimated time: 55 min

> Merges 📗 *Modules and Packages* (+ the `00-Modules_and_Packages` and `01-Name_and_Main` example folders) with 📘 *Modules and Pip*.

### 🎯 What you will learn
- What modules and packages are, and why we split code into files
- All forms of `import`; aliases; what to avoid (`import *`)
- Exploring modules with `dir()` and `help()`
- Writing your own module and package (`__init__.py`, sub-packages, `__all__`)
- Absolute vs relative imports; how Python finds modules (`sys.path`)
- `__name__ == "__main__"`, scripts vs importable modules
- Running modules with `python -m`
- 🆕 Explicit **lazy imports** (Python 3.15)

## 1. Vocabulary

| Term | Meaning |
|---|---|
| **Module** | a single `.py` file containing Python code |
| **Package** | a **folder** of modules with an `__init__.py` file |
| **Library** | a collection of packages you install (e.g. `requests`) |
| **Standard library** | modules that ship with Python (`math`, `os`, `json`, ...) |

## 2. Importing

In [1]:
import math                       # import the whole module
print(math.sqrt(16), math.ceil(2.4))

from math import pi, floor        # import specific names
print(pi, floor(3.7))

import datetime as dt             # alias
print(dt.date(2026, 10, 20))

from collections import Counter as C   # alias for a name
print(C("banana"))

4.0 3
3.141592653589793 3
2026-10-20
Counter({'a': 3, 'n': 2, 'b': 1})


> ❌ **Avoid `from module import *`.** It dumps every name into your namespace, hides where names come from, and can silently overwrite your variables (or built-ins).

## 3. Exploring a module (📗)

In [2]:
print(dir(math)[:20], "...")
help(math.ceil)

['__doc__', '__loader__', '__name__', '__package__', '__spec__', 'acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb', 'copysign', 'cos', 'cosh', 'degrees', 'dist'] ...
Help on built-in function ceil in module math:

ceil(x, /)
    Return the ceiling of x as an Integral.

    This is the smallest integer >= x.



## 4. Writing your own module

Any `.py` file is a module. Let's create one (`%%writefile` saves the cell as a file):

In [3]:
%%writefile mymodule.py
"""A tiny example module."""

PI_ISH = 3.14

def my_func():
    print("Hey I am in mymodule.py")

def greet(name):
    return f"Hello, {name}!"

Writing mymodule.py


In [4]:
import mymodule
mymodule.my_func()
print(mymodule.greet("Alice"), mymodule.PI_ISH)
print(mymodule.__doc__, mymodule.__file__.split("/")[-1])

Hey I am in mymodule.py
Hello, Alice! 3.14
A tiny example module. mymodule.py


## 5. Packages (📗 `MyMainPackage` example)

```
myprogram.py
MyMainPackage/
├── __init__.py
├── some_main_script.py
└── SubPackage/
    ├── __init__.py
    └── mysubscript.py
```

In [5]:
import os
os.makedirs("MyMainPackage/SubPackage", exist_ok=True)
files = {
    "MyMainPackage/__init__.py": "",
    "MyMainPackage/some_main_script.py": "def report_main():\n    print('Hey I am in some_main_script in main package.')\n",
    "MyMainPackage/SubPackage/__init__.py": "",
    "MyMainPackage/SubPackage/mysubscript.py": "def sub_report():\n    print('Hey Im a function inside mysubscript')\n",
}
for path, content in files.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
print("created package")

created package


In [6]:
from MyMainPackage import some_main_script
from MyMainPackage.SubPackage import mysubscript
from MyMainPackage.SubPackage.mysubscript import sub_report

some_main_script.report_main()
mysubscript.sub_report()
sub_report()

Hey I am in some_main_script in main package.
Hey Im a function inside mysubscript
Hey Im a function inside mysubscript


### What is `__init__.py` for?
- Marks a folder as a regular package
- Runs when the package is first imported, so it can set up the package
- Can **re-export** names for a cleaner API, and define `__all__` (what `import *` exposes)

In [7]:
with open("MyMainPackage/__init__.py", "w", encoding="utf-8") as f:
    f.write('"""Main package."""\nfrom .some_main_script import report_main\n__all__ = ["report_main"]\n__version__ = "1.0.0"\n')

import importlib, MyMainPackage
importlib.reload(MyMainPackage)              # reload because it was already imported
MyMainPackage.report_main()                  # thanks to the re-export in __init__.py
print(MyMainPackage.__version__)

Hey I am in some_main_script in main package.
1.0.0


> 💡 Since Python 3.3, folders *without* `__init__.py` can also be imported (**namespace packages**), but for your own projects always include `__init__.py`.

### Absolute vs relative imports
Inside a package, modules can import siblings **relatively**:
```python
from . import some_main_script          # same package
from .SubPackage import mysubscript     # sub-package
from .. import something                # parent package
```
Relative imports only work inside packages (not in a script you run directly). Absolute imports (`from MyMainPackage.SubPackage import ...`) are clearer and usually preferred.

## 6. How Python finds modules: `sys.path`

In [8]:
import sys
for p in sys.path[:5]:
    print(p or "(current directory)")

/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none/lib/python315.zip
/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none/lib/python3.15
/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none/lib/python3.15/lib-dynload
(current directory)
/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/lib/python3.15/site-packages


Search order: the script's folder / current directory → `PYTHONPATH` → standard library → `site-packages` (installed packages).

❌ **Classic bug:** naming your file `random.py`, `math.py`, `json.py` or `email.py` **shadows** the standard-library module of the same name. Never name files after modules you use.

Modules are cached in `sys.modules`; importing twice doesn't re-run the file:

In [9]:
import mymodule            # no re-execution
print("mymodule" in sys.modules)

True


## 7. `if __name__ == "__main__":` (📗 `01-Name_and_Main`)

Every module has a built-in variable `__name__`:
- When the file is **run directly** (`python one.py`), `__name__ == "__main__"`
- When it's **imported**, `__name__` is the module's name (`"one"`)

This lets a file be both an importable module **and** a runnable script.

In [10]:
%%writefile one.py
def func():
    print("FUNC() IN ONE.PY")

print("TOP LEVEL IN ONE.PY")

if __name__ == "__main__":
    print("ONE.PY is being run directly")
else:
    print("ONE.PY has been imported")

Writing one.py


In [11]:
%%writefile two.py
import one

print("TOP LEVEL IN TWO.PY")
one.func()

if __name__ == "__main__":
    print("TWO.PY is being run directly")
else:
    print("TWO.PY has been imported")

Writing two.py


In [12]:
import subprocess, sys
print(subprocess.run([sys.executable, "one.py"], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "two.py"], capture_output=True, text=True).stdout)

TOP LEVEL IN ONE.PY
ONE.PY is being run directly

TOP LEVEL IN ONE.PY
ONE.PY has been imported
TOP LEVEL IN TWO.PY
FUNC() IN ONE.PY
TWO.PY is being run directly



### ✅ The standard script layout

```python
"""Module docstring."""
import sys

def main() -> int:
    # program logic here
    return 0

if __name__ == "__main__":
    sys.exit(main())
```
Put logic in functions, call `main()` only under the guard, so importing the file (e.g. in tests) has no side effects.

## 8. Running modules with `-m`

```bash
python -m pip install requests     # run pip for THIS Python (recommended over plain `pip`)
python -m venv .venv               # create a virtual environment
python -m http.server 8000         # serve the current folder over HTTP
python -m json.tool data.json      # pretty-print JSON
python -m timeit "sum(range(100))" # time a snippet
python -m mypackage.tool           # run a module inside your package
```

## 9. 🆕 Explicit lazy imports (Python 3.15, PEP 810)

Big programs (and CLIs) can start slowly because every `import` at the top runs immediately. With the `lazy` soft keyword, a module is only loaded **the first time you use it**:

In [13]:
%%writefile heavy.py
print("   (heavy.py is loading... imagine 2 seconds of work)")
VALUE = 42

Writing heavy.py


In [14]:
lazy import heavy
print("After `lazy import`: is heavy loaded?", "heavy" in sys.modules)
print("Now using it →", heavy.VALUE)
print("Loaded now?", "heavy" in sys.modules)

After `lazy import`: is heavy loaded? False
   (heavy.py is loading... imagine 2 seconds of work)
Now using it → 42
Loaded now? True


Rules:
- Allowed only at **module (top) level**. Not inside functions, classes, or `try`/`except`/`with` blocks.
- Works for `lazy from pathlib import Path` too; `lazy from x import *` is not allowed.
- Errors such as a missing module surface at **first use**, not at the import line.
- Global switches: `python -X lazy_imports=all` or env var `PYTHON_LAZY_IMPORTS`.
- On older Pythons this syntax is a `SyntaxError`; libraries can use `__lazy_modules__ = ["json"]` to opt in compatibly.

## 10. A quick tour of the standard library

| Area | Modules |
|---|---|
| Text | `string`, `re`, `textwrap`, `difflib` |
| Data structures | `collections`, `heapq`, `bisect`, `array`, `enum`, `dataclasses` |
| Math | `math`, `cmath`, `decimal`, `fractions`, `random`, `statistics` |
| Functional | `itertools`, `functools`, `operator` |
| Files & OS | `os`, `pathlib`, `shutil`, `glob`, `tempfile`, `zipfile`, `tarfile`, 🆕 `compression` (3.14) |
| Data formats | `json`, `csv`, `pickle`, `sqlite3`, `tomllib`, `xml`, `configparser` |
| Time | `time`, `datetime`, `zoneinfo`, `calendar` |
| Concurrency | `threading`, `multiprocessing`, `concurrent.futures`, `asyncio`, 🆕 `concurrent.interpreters` (3.14) |
| Internet | `urllib`, `http`, `email`, `smtplib`, `imaplib`, `socket` |
| Dev tools | `unittest`, `doctest`, `pdb`, `logging`, `timeit`, 🆕 `profiling` (3.15), `typing`, `argparse` |

> ➡️ Next in this module: pip & venv (02), the complete `pyproject.toml` (03), **uv** (04), **Poetry** (05), building wheels (06), publishing with twine/uv/Poetry (07), CLI apps (08), choosing tools (09).

## 📝 Summary
- Module = `.py` file; package = folder with `__init__.py`.
- `import x`, `from x import y`, `import x as z`; avoid `import *`.
- `if __name__ == "__main__":` separates "run as script" from "imported".
- Python searches `sys.path`; don't shadow stdlib names.
- 🆕 `lazy import x` (3.15) defers loading until first use.

## 🏋️ Practice
1. Create `geometry.py` with `circle_area(r)` and `square_area(s)`; import and use both.
2. Turn it into a package `shapes/` with modules `circle.py` and `square.py`, and re-export both functions from `shapes/__init__.py`.
3. Add a `main()` with a `__name__` guard to `geometry.py` that prints sample results.

<details><summary>Show solutions</summary>

```python
# geometry.py
import math
def circle_area(r): return math.pi * r ** 2
def square_area(s): return s * s
def main():
    print(circle_area(1), square_area(2))
if __name__ == "__main__":
    main()

# shapes/__init__.py
from .circle import circle_area
from .square import square_area
__all__ = ["circle_area", "square_area"]
```
</details>